In [166]:
import pandas as pd
import numpy as np
import pickle
import json

df = pd.read_csv('anilist_anime_data_complete.csv')

In [167]:
df = df[df['popularity'] > 30000]
df = df[df['isAdult'] == False]
df = df.reset_index()
df = df[['title_romaji', 'id', 'episodes', 'duration', 'genres', 'tags']]

In [168]:
for tags in range(len(df['tags'].apply(json.loads))):
    for tag in df['tags'].apply(json.loads)[tags]:
        if tag['rank'] > 75 and tag['isAdult'] == False:
            if tag['name'] not in df.columns:
                df[tag['name']] = 0
            df[tag['name']][tags] = tag['rank']

C:\Users\Aryan\AppData\Local\Temp\ipykernel_28788\1220729427.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[tag['name']][tags] = tag['rank']
C:\Users\Aryan\AppData\Local\Temp\ipykernel_28788\1220729427.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[tag['name']][tags] = tag['rank']
C:\Users\Aryan\AppData\Local\Temp\ipykernel_28788\1220729427.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[tag['name']][tags] = tag[

In [169]:
for genres in range(len(df['genres'].apply(json.loads))):
    for genre in df['genres'].apply(json.loads)[genres]:
            if genre not in df.columns:
                df[genre] = 0
            df[genre][genres] = 1

C:\Users\Aryan\AppData\Local\Temp\ipykernel_28788\906486332.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[genre] = 0
C:\Users\Aryan\AppData\Local\Temp\ipykernel_28788\906486332.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[genre][genres] = 1
C:\Users\Aryan\AppData\Local\Temp\ipykernel_28788\906486332.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `ne

In [170]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

In [171]:
cosine_sim = linear_kernel(df.iloc[:, 6:], df.iloc[:, 6:])

In [172]:
np.save('matrix.npy', cosine_sim)

In [173]:
import numpy as np
with open('matrix.npy', 'rb') as f:
    cosine_sim = np.load(f)
cosine_sim

array([[4.3462e+04, 1.4160e+04, 1.3841e+04, ..., 6.7210e+03, 6.9620e+03,
        6.3230e+03],
       [1.4160e+04, 6.2357e+04, 0.0000e+00, ..., 1.8888e+04, 6.9600e+03,
        1.5239e+04],
       [1.3841e+04, 0.0000e+00, 9.5694e+04, ..., 1.9807e+04, 2.0000e+00,
        2.0000e+00],
       ...,
       [6.7210e+03, 1.8888e+04, 1.9807e+04, ..., 7.5438e+04, 2.2704e+04,
        6.6390e+03],
       [6.9620e+03, 6.9600e+03, 2.0000e+00, ..., 2.2704e+04, 8.5140e+04,
        6.8770e+03],
       [6.3230e+03, 1.5239e+04, 2.0000e+00, ..., 6.6390e+03, 6.8770e+03,
        6.7298e+04]])

In [174]:
indices = pd.Series(df.index, index=df['title_romaji']).drop_duplicates()

In [175]:
df.to_csv("indices.csv", index=True)

In [176]:
def get_recommendations(title, a_list, cosine_sim=cosine_sim, num_recommend = 10):
    if title in df['title_romaji'].values:
        num_recd = 0
        raw_recd = 1
        top_similar = []
        idx = indices[title]
        sim_scores = list(enumerate(cosine_sim[idx]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        while num_recd <= num_recommend:
            top_rec = sim_scores[raw_recd]
            if df['title_romaji'].iloc[top_rec[0]] not in a_list and "Season" not in df['title_romaji'].iloc[top_rec[0]]:
                top_similar.append(sim_scores[raw_recd])
                num_recd += 1
            raw_recd += 1
        movie_indices = [i[0] for i in top_similar]
        return df['title_romaji'].iloc[movie_indices]

In [177]:
import requests
import json
query = '''
query ($type: MediaType!, $userName: String!) {
  MediaListCollection(type: $type, userName: $userName) {
    lists {
      name
      entries {
        id
        media {
          title {
            romaji
          }
        }
      }
    }
  }
}
'''

variables = {
    'type': 'ANIME',
    'userName':'aryantestlist'
}

url = 'https://graphql.anilist.co'

response = requests.post(url, json={'query': query, 'variables': variables})

In [178]:
json_data = json.loads(response.content)
anime_list = []
for j in json_data['data']['MediaListCollection']['lists']:
    if j['name'] != "Dropped" and j['name'] != "Planning":
        for show in j['entries']:
            anime_list.append(str(show['media']['title']['romaji']))

In [179]:
for show in anime_list:
    if show not in df['title_romaji'].values:
        print(show)
        anime_list.remove(show)

Jujutsu Kaisen: Shimetsu Kaiyuu - Zenpen
Shoushimin Series 2nd Season
Kill Ao


In [180]:
total_recs = {}
for title in anime_list:  
    recs = list(get_recommendations(title, anime_list, num_recommend = 10))
    for rec in recs:
        if rec not in anime_list:
            if rec in total_recs:
                total_recs[rec] += recs.index(rec)
            else:
                total_recs[rec] = recs.index(rec)

In [182]:
dict(sorted(total_recs.items(), key=lambda item: item[1], reverse=True))

{'Tokyo Revengers: Tenjiku-hen': 54,
 'VINLAND SAGA': 38,
 'Bakuten!!': 35,
 'Shinsekai yori': 33,
 'Hagane no Renkinjutsushi: FULLMETAL ALCHEMIST': 32,
 'Code Geass: Hangyaku no Lelouch R2': 30,
 'Summer Time Render': 27,
 'Higurashi no Naku Koro ni Gou': 27,
 'Free!': 26,
 'Enen no Shouboutai: San no Shou': 26,
 'Tougen Anki': 25,
 'PLUTO': 25,
 'CLANNAD': 25,
 'ONE PIECE': 24,
 'Tokyo Revengers': 23,
 'Yuuki Yuuna wa Yuusha de Aru': 22,
 'Sentai Daishikkaku': 22,
 'Ima, Soko ni Iru Boku': 21,
 'Ginga Eiyuu Densetsu': 20,
 'Shakunetsu Kabaddi': 19,
 'Kuroko no Basket 2nd SEASON': 19,
 'Eyeshield 21': 19,
 'Gachiakuta': 19,
 'Shin Seiki Evangelion Movie: Air / Magokoro wo, Kimi ni': 19,
 'Haikyuu!! Riku VS Kuu': 18,
 'Katekyou Hitman REBORN!': 18,
 'SANDA': 18,
 'Kimetsu no Yaiba: Mugenjou-hen Movie 1 - Akaza Sairai': 18,
 'Magi: The kingdom of magic': 18,
 'Inuyashiki': 17,
 'Overlord III': 17,
 'Go-toubun no Hanayome Movie': 17,
 'Diamond no Ace': 16,
 'Sakigake!! Cromartie Koukou':

In [190]:
list(dict(sorted(total_recs.items(), key=lambda item: item[1], reverse=True)).keys())[0:10]

['Tokyo Revengers: Tenjiku-hen',
 'VINLAND SAGA',
 'Bakuten!!',
 'Shinsekai yori',
 'Hagane no Renkinjutsushi: FULLMETAL ALCHEMIST',
 'Code Geass: Hangyaku no Lelouch R2',
 'Summer Time Render',
 'Higurashi no Naku Koro ni Gou',
 'Free!',
 'Enen no Shouboutai: San no Shou']

In [219]:
show_list = list(dict(sorted(total_recs.items(), key=lambda item: item[1], reverse=True)).keys())[0:10]
ids = []
for j in show_list:
    ids.append(indices[indices['title_romaji'] == j]['id'].values[0])
json_response = []
for show in range(len(show_list)):
    json_response.append({ids[show]: show_list[show]})

In [220]:
json_response

[{'Tokyo Revengers: Tenjiku-hen': 163329},
 {'VINLAND SAGA': 101348},
 {'Bakuten!!': 125839},
 {'Shinsekai yori': 13125},
 {'Hagane no Renkinjutsushi: FULLMETAL ALCHEMIST': 5114},
 {'Code Geass: Hangyaku no Lelouch R2': 2904},
 {'Summer Time Render': 129201},
 {'Higurashi no Naku Koro ni Gou': 114446},
 {'Free!': 18507},
 {'Enen no Shouboutai: San no Shou': 149118}]

In [210]:
print(indices[indices['title_romaji'] == 'VINLAND SAGA']['id'].values[0])

101348


In [200]:
indices

,Unnamed: 0,title_romaji,id,episodes,duration,genres,tags,Boxing,Coming of Age,Delinquents,...,Romance,Mystery,Fantasy,Horror,Supernatural,Psychological,Music,Ecchi,Mahou Shoujo,Thriller
0,0,Ashita no Joe,2402,79.0,25.0,"[""Action"", ""Drama"", ""Slice of Life"", ""Sports""]","[{""id"": 169, ""name"": ""Boxing"", ""description"": ...",90,90,87,...,0,0,0,0,0,0,0,0,0,0
1,1,Lupin III,1412,23.0,25.0,"[""Adventure"", ""Comedy""]","[{""id"": 50, ""name"": ""Seinen"", ""description"": ""...",0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2,Kidou Senshi Gundam,80,43.0,25.0,"[""Drama"", ""Mecha"", ""Sci-Fi""]","[{""id"": 111, ""name"": ""War"", ""description"": ""Pa...",0,80,0,...,0,0,0,0,0,0,0,0,0,0
3,3,Versailles no Bara,338,40.0,23.0,"[""Adventure"", ""Drama"", ""Romance""]","[{""id"": 25, ""name"": ""Historical"", ""description...",0,0,0,...,1,0,0,0,0,0,0,0,0,0
4,4,Lupin III: Cagliostro no Shiro,1430,1.0,100.0,"[""Action"", ""Adventure"", ""Comedy"", ""Mystery""]","[{""id"": 648, ""name"": ""Crime"", ""description"": ""...",0,0,0,...,0,1,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2137,2137,Sozai Saishuka no Isekai Ryokouki,187663,12.0,24.0,"[""Adventure"", ""Comedy"", ""Fantasy"", ""Slice of L...","[{""id"": 244, ""name"": ""Isekai"", ""description"": ...",0,0,0,...,0,0,1,0,0,0,0,0,0,0
2138,2138,Mikata ga Yowasugite Hojo Mahou ni Tesshiteita...,188487,12.0,23.0,"[""Action"", ""Adventure"", ""Fantasy""]","[{""id"": 29, ""name"": ""Magic"", ""description"": ""P...",0,0,0,...,0,0,1,0,0,0,0,0,0,0
2139,2139,Dr. STONE: SCIENCE FUTURE Part 2,189117,12.0,24.0,"[""Action"", ""Adventure"", ""Comedy"", ""Sci-Fi""]","[{""id"": 93, ""name"": ""Post-Apocalyptic"", ""descr...",0,0,0,...,0,0,0,0,0,0,0,0,0,0
2140,2140,Towa no Yuugure,195153,13.0,24.0,"[""Action"", ""Drama"", ""Romance"", ""Sci-Fi""]","[{""id"": 175, ""name"": ""Robots"", ""description"": ...",0,0,0,...,1,0,0,0,0,0,0,0,0,0
